In [1]:
from Trips_Extractor import extract_trips_from_pdf
from Lines_Extractor import parse_line_report_pdf
from master_lines_creation import creating_master_line
from pprint import pprint
import Processing_fucntions as pf
import pandas as pd
from pprint import pprint

In [2]:
lines_pdf = "/home/aleluc/Github_repos/UPS-project/SamplePDFs/2304 Lines edge case.pdf"
lines = parse_line_report_pdf(lines_pdf, first_calendar_page=1)
bid_period_info = {x: lines[x] for x in ('bid_period_date_range','pay_period_date_ranges')}

In [3]:
trips_pdf = "/home/aleluc/Github_repos/UPS-project/SamplePDFs/2304 Trips.pdf"
trips = extract_trips_from_pdf(trips_pdf)

In [4]:
master_lines = creating_master_line(trips, lines)

In [5]:
pf.add_blockiness_scores(master_lines, bid_period_info['bid_period_date_range']['start'], bid_period_info['bid_period_date_range']['end'])
pf.add_company_ticket_percentages(master_lines)
pf.add_training_fit_score(master_lines, "2023-07-06", "2023-07-10",bid_period_info['bid_period_date_range']['start'],bid_period_info['bid_period_date_range']['end'])
vacation_ranges=[{"start": "2023-05-01", "end": "2023-05-15"},{"start": "2023-07-01", "end": "2023-07-08"}]
pf.add_vacation_days_off_score(master_lines,vacation_ranges, bid_period_info,pp_drop_threshold_days=14,save_details=False)

In [6]:
pprint(master_lines)

{1: {'PPs': [{'BT': '52:48',
              'CT': '72:00',
              'DD': 12,
              'DO': '11',
              'assignments': [{'flights': [{'arrival': 'MIA',
                                            'departure': 'SDF',
                                            'end_date': '2023-05-23',
                                            'rest': None,
                                            'route_flags': None,
                                            'start_date': '2023-05-23'},
                                           {'arrival': 'SDF',
                                            'departure': 'MIA',
                                            'end_date': '2023-05-23',
                                            'rest': None,
                                            'route_flags': None,
                                            'start_date': '2023-05-23'}],
                               'premium': 0.0,
                               'tafb': '7h58',
             

In [28]:
from datetime import date, datetime, timedelta
from collections import defaultdict
import pandas as pd


def _to_date(value):
    if value is None or value == "":
        return None

    if isinstance(value, datetime):
        return value.date()

    if isinstance(value, date):
        return value

    return datetime.strptime(str(value), "%Y-%m-%d").date()


def _date_range_inclusive(start, end):
    start = _to_date(start)
    end = _to_date(end)

    if start is None or end is None:
        return []

    if end < start:
        start, end = end, start

    return [
        start + timedelta(days=i)
        for i in range((end - start).days + 1)
    ]


def _assignment_label(assignment):
    """
    Creates the text that will appear inside each calendar-date cell.
    """

    # Current master_lines format
    if assignment.get("trip_id") is not None:
        return f"T{assignment['trip_id']}"

    # Older/raw lines format, if still used somewhere
    assignment_type = assignment.get("type")
    value = assignment.get("value")

    if assignment_type == "trip" and value is not None:
        return f"T{value}"

    if assignment_type and value is not None:
        return f"{str(assignment_type).upper()} {value}"

    if assignment_type:
        return str(assignment_type).upper()

    return ""


def _assignment_dates(assignment):
    """
    Returns all calendar dates touched by one assignment/trip.

    For your current master_lines, this looks inside assignment["flights"].
    """

    flights = assignment.get("flights") or []
    flight_dates = []

    for flight in flights:
        start = _to_date(flight.get("start_date"))
        end = _to_date(flight.get("end_date"))

        if start and end:
            flight_dates.extend(_date_range_inclusive(start, end))
        elif start:
            flight_dates.append(start)
        elif end:
            flight_dates.append(end)

    if flight_dates:
        # Important for multi-day trips:
        # fills every calendar day from first flight date to last flight date.
        return _date_range_inclusive(min(flight_dates), max(flight_dates))

    # Fallback for older/raw format
    assignment_date = _to_date(assignment.get("date"))
    if assignment_date:
        return [assignment_date]

    return []


def master_lines_to_dataframe(
    master_lines,
    bid_start,
    bid_end,
    *,
    off_value="",
    date_col_format="%Y-%m-%d",
):
    """
    Converts master_lines into one pandas row per line.

    Result columns:
        - line_number
        - extra_vacation_days
        - training_fit_score
        - blockiness_score
        - tot_DO
        - company_ticket_pct
        - DO_start_bid
        - DO_end_bid
        - one column for every calendar date in the bid period

    Calendar cells:
        - trip days show T7, T46, etc.
        - days off are blank by default
        - use off_value="DO" if you want days off visibly marked
    """

    bid_dates = _date_range_inclusive(bid_start, bid_end)
    rows = []

    for line_number, line_data in master_lines.items():

        worked_by_date = defaultdict(list)

        for pp in line_data.get("PPs", line_data.get("pay_periods", [])):
            for assignment in pp.get("assignments", []):
                label = _assignment_label(assignment)

                for d in _assignment_dates(assignment):
                    if bid_dates[0] <= d <= bid_dates[-1]:
                        worked_by_date[d].append(label)

        date_values = {}

        for d in bid_dates:
            labels = [label for label in worked_by_date.get(d, []) if label]
            date_values[d] = "/".join(labels) if labels else off_value

        row = {
            "line_number": line_number,
            "extra_vacation_days": line_data.get("extra_vacation_days", 0),
            "training_fit_score": line_data.get("training_fit_score", 0),
            "blockiness_score": line_data.get("blockiness_score", 0),
            "tot_DO": line_data.get("tot_DO", 0),
            "company_ticket_pct": line_data.get("company_ticket_pct", 0),
        }

        for d in bid_dates:
            row[d.strftime(date_col_format)] = date_values[d]

        rows.append(row)

    df = pd.DataFrame(rows)

    if not df.empty:
        df = df.sort_values("line_number").reset_index(drop=True)

    return df

df = master_lines_to_dataframe(
    master_lines,
    bid_start=bid_period_info['bid_period_date_range']['start'],
    bid_end=bid_period_info['bid_period_date_range']['end'],
    off_value="",          # or "DO" if you want day-off cells marked
)



In [29]:
df

,line_number,extra_vacation_days,training_fit_score,blockiness_score,tot_DO,company_ticket_pct,2023-05-21,2023-05-22,2023-05-23,2023-05-24,2023-05-25,2023-05-26,2023-05-27,2023-05-28,2023-05-29,2023-05-30,2023-05-31,2023-06-01,2023-06-02,2023-06-03,2023-06-04,2023-06-05,2023-06-06,2023-06-07,2023-06-08,2023-06-09,2023-06-10,2023-06-11,2023-06-12,2023-06-13,2023-06-14,2023-06-15,2023-06-16,2023-06-17,2023-06-18,2023-06-19,2023-06-20,2023-06-21,2023-06-22,2023-06-23,2023-06-24,2023-06-25,2023-06-26,2023-06-27,2023-06-28,2023-06-29,2023-06-30,2023-07-01,2023-07-02,2023-07-03,2023-07-04,2023-07-05,2023-07-06,2023-07-07,2023-07-08,2023-07-09,2023-07-10,2023-07-11,2023-07-12,2023-07-13,2023-07-14,2023-07-15,2023-07-16
0,1,2,29.3,47.482517,24,2.1,,,T7,T7,T14,,,,,,T8,T8,,,,,T8,T8,T8,T34,,,,T8,T8,T8,,,,,T22,T8,T8,,,,,T8,T8,T8,,,,,,,T9,,,,,T9,T9,T9,T34,T46,
1,2,2,45.3,59.150641,25,0.0,,,T4,T4,T4,T4,T31,,,,,,,,,,T5,T5,T5,,,,,T5,T5,T5,T65,,,,,,,,,,,T5,T5,T5,T5,,,,,T49,T3,T65,,,,T6,T6,T6,T6,,
2,37,0,26.0,102.946387,28,62.5,T291,T291,T291,T291,T291,T291,,,,,,,,,,,,,,,,T356,T356,T356,T356,T356,T356,T356,,,,,,,,T396,T396,T396,T396,T396,T396,T396,T396,,,,,,,,T309,T309,T309,T309,T309,T309,
3,38,0,5.2,85.845353,31,75.0,T307,T307,T307,T307,T307,T307,T307,,,,,,,,T400,T400,T400,T400,T400,T400,T400,T400,,,,,,,T326,T326,T326,T326,T326,T326,T326,,T338,T338,T338,T338,T338,T338,T338,,,,,,,,,,,,,,
4,105,31,63.0,71.840278,16,37.5,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,T600,T600,T600,,,,,,T600,T600,T600,,,,,T601,T601,T601,,,,,,T612,T612,T612
5,106,0,9.0,87.115385,13,75.0,T395,T395,T395,T395,T395,T395,T395,T395,,,,,,,,,,,,,,T310,T310,T310,T310,T310,T310,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
6,135,56,14.3,60.000000,0,0.0,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
7,136,0,14.3,7.000000,0,0.0,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
8,147,2,14.3,35.330357,28,0.0,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
9,148,1,14.3,29.517857,28,0.0,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,


In [30]:
from datetime import date, datetime, timedelta
from collections import defaultdict
import pandas as pd


def _to_date(value):
    if value is None or value == "":
        return None

    if isinstance(value, datetime):
        return value.date()

    if isinstance(value, date):
        return value

    return datetime.strptime(str(value), "%Y-%m-%d").date()


def _date_range_inclusive(start, end):
    start = _to_date(start)
    end = _to_date(end)

    if start is None or end is None:
        return []

    if end < start:
        start, end = end, start

    return [
        start + timedelta(days=i)
        for i in range((end - start).days + 1)
    ]


def _format_rest(rest):
    """
    Converts rest value into the text used inside [CITYrest].

    Examples:
        48       -> "48"
        "48"     -> "48"
        "48h00"  -> "48"
        "12h30"  -> "12.5"
        "12:30"  -> "12.5"
    """

    if rest is None:
        return ""

    if isinstance(rest, (int, float)):
        if float(rest).is_integer():
            return str(int(rest))
        return str(round(float(rest), 1))

    text = str(rest).strip()

    if "h" in text:
        hours, minutes = text.split("h", 1)
        hours = int(hours)
        minutes = int(minutes or 0)

        value = hours + minutes / 60

        if value.is_integer():
            return str(int(value))

        return str(round(value, 1))

    if ":" in text:
        hours, minutes = text.split(":", 1)
        hours = int(hours)
        minutes = int(minutes or 0)

        value = hours + minutes / 60

        if value.is_integer():
            return str(int(value))

        return str(round(value, 1))

    return text


def _format_route_flags(route_flags):
    """
    Converts route_flags to:
        None              -> ""
        ["DH AA"]         -> "(DH AA)"
        ["c", "IRO"]      -> "(c,IRO)"
    """

    if not route_flags:
        return ""

    if isinstance(route_flags, str):
        flags = [route_flags]
    else:
        flags = list(route_flags)

    flags = [str(flag).strip() for flag in flags if str(flag).strip()]

    if not flags:
        return ""

    return f"({','.join(flags)})"


def _arrival_text(arrival, rest=None, close_trip=False):
    """
    Formats the arrival side of a flight.

    Examples:
        arrival="DFW", rest=None       -> "DFW"
        arrival="DFW", rest=48         -> "[DFW48] "
        arrival="SDF", close_trip=True -> "SDF}"
    """

    if rest is not None:
        text = f"[{arrival}{_format_rest(rest)}] "
    else:
        text = str(arrival)

    if close_trip:
        text = text.rstrip() + "}"

    return text


def _append_trip_piece(parts_by_date, d, piece):
    """
    Appends text to one date for one trip.

    This keeps the route readable when multiple pieces occur on the same day.
    """

    if not piece:
        return

    existing = parts_by_date[d]

    if not existing:
        parts_by_date[d] = piece
        return

    # If the previous piece already intentionally ended with a space,
    # do not add another separator.
    if existing.endswith(" "):
        parts_by_date[d] = existing + piece
    else:
        parts_by_date[d] = existing + " " + piece


def _render_trip_by_date(assignment):
    """
    Converts one trip assignment into a dictionary:

        {
            date(2023, 6, 1): "{105SDF-ONT",
            date(2023, 6, 2): "[DFW12] DFW-",
            date(2023, 6, 3): "SDF}",
        }

    Rules:
        - first flight starts with {trip_id
        - final flight arrival ends with }
        - travel is shown with -
        - route flags go after the dash and before arrival city
        - rest is shown as [CITYrest]
        - empty full calendar days caused by rest are shown as *
    """

    trip_id = assignment.get("trip_id")
    flights = assignment.get("flights") or []

    parts_by_date = defaultdict(str)

    if not flights:
        return parts_by_date

    for index, flight in enumerate(flights):
        dep = flight.get("departure")
        arr = flight.get("arrival")

        start_date = _to_date(flight.get("start_date"))
        end_date = _to_date(flight.get("end_date"))

        if start_date is None and end_date is None:
            continue

        if start_date is None:
            start_date = end_date

        if end_date is None:
            end_date = start_date

        flags = _format_route_flags(flight.get("route_flags"))
        rest = flight.get("rest")

        is_first_flight = index == 0
        is_last_flight = index == len(flights) - 1

        trip_open = f"{{{trip_id}" if is_first_flight else ""

        arrival = _arrival_text(
            arr,
            rest=rest,
            close_trip=is_last_flight,
        )

        # Normal same-day flight
        if start_date == end_date:
            piece = f"{trip_open}{dep}-{flags}{arrival}"
            _append_trip_piece(parts_by_date, start_date, piece)

        # Flight departs one date and arrives on another date
        else:
            departure_piece = f"{trip_open}{dep}-"
            arrival_piece = f"{flags}{arrival}"

            _append_trip_piece(parts_by_date, start_date, departure_piece)
            _append_trip_piece(parts_by_date, end_date, arrival_piece)

        # Mark full empty rest days between this flight and the next flight.
        if index < len(flights) - 1 and rest is not None:
            next_flight = flights[index + 1]
            next_start = _to_date(next_flight.get("start_date"))

            if next_start is not None:
                gap_start = end_date + timedelta(days=1)
                gap_end = next_start - timedelta(days=1)

                for gap_day in _date_range_inclusive(gap_start, gap_end):
                    if not parts_by_date[gap_day]:
                        parts_by_date[gap_day] = "*"

    return parts_by_date


def _count_start_days_off(date_values, bid_dates, off_value):
    count = 0

    for d in bid_dates:
        if date_values.get(d, off_value) == off_value:
            count += 1
        else:
            break

    return count


def _count_end_days_off(date_values, bid_dates, off_value):
    count = 0

    for d in reversed(bid_dates):
        if date_values.get(d, off_value) == off_value:
            count += 1
        else:
            break

    return count


def master_lines_to_dataframe(
    master_lines,
    bid_start,
    bid_end,
    *,
    off_value="",
    date_col_format="%Y-%m-%d",
):
    """
    Converts master_lines into one pandas row per line.

    Calendar cells:
        - trip start begins with {trip_id
        - trip end ends with }
        - route shown by city-to-city text
        - layovers shown as [CITYrest]
        - full empty layover/rest days shown as *
        - true days off are blank by default
    """

    bid_dates = _date_range_inclusive(bid_start, bid_end)
    rows = []

    for line_number, line_data in master_lines.items():

        trip_text_by_date = defaultdict(list)

        for pp in line_data.get("PPs", line_data.get("pay_periods", [])):
            for assignment in pp.get("assignments", []):
                rendered_trip = _render_trip_by_date(assignment)

                for d, text in rendered_trip.items():
                    if bid_dates[0] <= d <= bid_dates[-1]:
                        trip_text_by_date[d].append(text)

        date_values = {}

        for d in bid_dates:
            pieces = trip_text_by_date.get(d, [])

            if pieces:
                # If more than one trip touches the same calendar date,
                # separate them clearly.
                date_values[d] = " | ".join(pieces)
            else:
                date_values[d] = off_value

        row = {
            "line_number": line_number,
            "extra_vacation_days": line_data.get("extra_vacation_days", 0),
            "training_fit_score": line_data.get("training_fit_score", 0),
            "blockiness_score": line_data.get("blockiness_score", 0),
            "tot_DO": line_data.get("tot_DO", 0),
            "company_ticket_pct": line_data.get("company_ticket_pct", 0),
            "DO_start_bid": _count_start_days_off(date_values, bid_dates, off_value),
            "DO_end_bid": _count_end_days_off(date_values, bid_dates, off_value),
        }

        for d in bid_dates:
            row[d.strftime(date_col_format)] = date_values[d]

        rows.append(row)

    df = pd.DataFrame(rows)

    if not df.empty:
        df = df.sort_values("line_number").reset_index(drop=True)

    return df

In [31]:
df = master_lines_to_dataframe(
    master_lines,
    bid_start="2023-05-21",
    bid_end="2023-07-15",
    off_value="",
    date_col_format="%b %d",
)

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)  # Shows full text in cells
df

,line_number,extra_vacation_days,training_fit_score,blockiness_score,tot_DO,company_ticket_pct,DO_start_bid,DO_end_bid,May 21,May 22,May 23,May 24,May 25,May 26,May 27,May 28,May 29,May 30,May 31,Jun 01,Jun 02,Jun 03,Jun 04,Jun 05,Jun 06,Jun 07,Jun 08,Jun 09,Jun 10,Jun 11,Jun 12,Jun 13,Jun 14,Jun 15,Jun 16,Jun 17,Jun 18,Jun 19,Jun 20,Jun 21,Jun 22,Jun 23,Jun 24,Jun 25,Jun 26,Jun 27,Jun 28,Jun 29,Jun 30,Jul 01,Jul 02,Jul 03,Jul 04,Jul 05,Jul 06,Jul 07,Jul 08,Jul 09,Jul 10,Jul 11,Jul 12,Jul 13,Jul 14,Jul 15
0,1,2,29.3,47.482517,24,2.1,2,0,,,{7SDF-MIA MIA-SDF},{7SDF-MIA MIA-SDF},{14SDF-MCO MCO-SDF},,,,,,{8SDF-MIA MIA-SDF},{8SDF-MIA MIA-SDF},,,,,{8SDF-MIA MIA-SDF},{8SDF-MIA MIA-SDF},{8SDF-MIA MIA-SDF},{34SDF-PHL PHL-SDF},,,,{8SDF-MIA MIA-SDF},{8SDF-MIA MIA-SDF},{8SDF-MIA MIA-SDF},,,,,{22SDF-IAH IAH-SDF},{8SDF-MIA MIA-SDF},{8SDF-MIA MIA-SDF},,,,,{8SDF-MIA MIA-SDF},{8SDF-MIA MIA-SDF},{8SDF-MIA MIA-SDF},,,,,,,{9SDF-MIA MIA-SDF},,,,,{9SDF-MIA MIA-SDF},{9SDF-MIA MIA-SDF},{9SDF-MIA MIA-SDF},{34SDF-PHL PHL-SDF},{46SDF-DTW DTW-(DH DL)SDF}
1,2,2,45.3,59.150641,25,0.0,2,1,,,{4SDF-EWR EWR-SDF},{4SDF-EWR EWR-SDF},{4SDF-EWR EWR-SDF},{4SDF-EWR EWR-SDF},{31SDF-MKE MKE-SDF},,,,,,,,,,{5SDF-EWR EWR-SDF},{5SDF-EWR EWR-SDF},{5SDF-EWR EWR-SDF},,,,,{5SDF-EWR EWR-SDF},{5SDF-EWR EWR-SDF},{5SDF-EWR EWR-SDF},{65SDF-EWR EWR-SDF},,,,,,,,,,,{5SDF-EWR EWR-SDF},{5SDF-EWR EWR-SDF},{5SDF-EWR EWR-SDF},{5SDF-EWR EWR-SDF},,,,,{49SDF-JFK JFK-SDF},{3SDF-JFK JFK-SDF},{65SDF-EWR EWR-SDF},,,,{6SDF-EWR EWR-SDF},{6SDF-EWR EWR-SDF},{6SDF-EWR EWR-SDF},{6SDF-EWR EWR-SDF},
2,37,0,26.0,102.946387,28,62.5,0,0,{291SDF-,(DH DL)ATL ATL-(DH DL)[DFW20.9],* DFW-RFD RFD-[DFW15.4],* DFW-SDF SDF-[OAK25.2],* OAK-[ONT16.6],* ONT-DEN DEN-SDF},,,,,,,,,,,,,,,,{356SDF-,(DH AA)CLT CLT-(DH AA)[DFW20.5],* DFW-RFD RFD-[DFW15.8],* DFW-RFD RFD-[DFW15.4],* DFW-SDF SDF-[OAK25.2],* OAK-[ONT13.5],* ONT-(DH DL)ATL ATL-(DH DL)SDF},,,,,,,,{396SDF-(DH DL)DTW DTW-,[LAN24.1],* LAN-SDF SDF-[LAN15.8],* LAN-SDF SDF-[LAN15.8],* LAN-SDF SDF-[LAN15.8],* LAN-SDF SDF-[LAN15.8],* LAN-SDF SDF-[EWR17.4],* EWR-SDF},,,,,,,,{309SDF-(DH WN)MDW MDW-(DH WN)[MCI20.5],* MCI-SDF SDF-[MKE15.3],* MKE-SDF SDF-[MKE15.3],* MKE-SDF SDF-[MKE15.3],* MKE-SDF SDF-[MKE15.3],* MKE-SDF}
3,38,0,5.2,85.845353,31,75.0,0,13,{307SDF-,(DH WN)[ATL24.0],* ATL-SDF SDF-[CLE14.7],* CLE-SDF SDF-[CLE14.7],* CLE-SDF SDF-[CLE14.7],* CLE-SDF SDF-[CLE14.7],* CLE-SDF},,,,,,,,{400SDF-(DH DL)ATL ATL-(DH DL)[MDT27.8],*,MDT-SDF SDF-[MDT15.2],* MDT-SDF SDF-[MDT15.2],* MDT-SDF SDF-[MDT15.2],* MDT-SDF SDF-[MDT15.9],* MDT-SDF SDF-[MDT11.6] MDT-,* (DH AA)CLT CLT-(DH AA)SDF},,,,,,,{326SDF-(DH DL)[DTW26],*,DTW-SDF SDF-[ATL16.9],* ATL-SDF SDF-[ATL16.9],* ATL-SDF SDF-[ATL16.9],* ATL-SDF SDF-[MIA14.7],* MIA-SDF},,{338SDF-(DH AA)[DFW14.8],* DFW-[PHL31.6],* PHL-,SDF SDF-[ORD16.6],* ORD-SDF SDF-[MCO14.6],* MCO-SDF SDF-[CLE12.6] CLE-,* (DH WN)MDW MDW-(DH WN)SDF},,,,,,,,,,,,,
4,105,31,63.0,71.840278,16,37.5,31,0,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,{600SDF-,[HNL20.6],* HNL-(DH DL)ATL ATL-(DH DL)SDF},,,,,,{600SDF-,[HNL20.6],* HNL-(DH DL)ATL ATL-(DH DL)SDF},,,,,{601SDF-,[HNL20.6],* HNL-(DH DL)ATL ATL-(DH DL)SDF},,,,,,"{612SDF-(C,DH UPS)CGN CGN-[FRA25.1]",* FRA-
5,106,0,9.0,87.115385,13,75.0,0,28,{395SDF-(DH DL)[ATL25.5],*,ATL-SDF SDF-[ATL16.9],* ATL-SDF SDF-[ATL16.9],* ATL-SDF SDF-[ATL16.9],* ATL-SDF SDF-[MCO14.6],* MCO-SDF SDF-[EWR17.4],* EWR-SDF},,,,,,,,,,,,,,{310SDF-,(DH AA)CLT CLT-(DH AA)[RDU20.0],* RDU-SDF SDF-[RDU14.5],* RDU-SDF SDF-[RDU14.5],* RDU-SDF SDF-[RDU14.5],* RDU-SDF SDF-[RDU9.8] RDU-(DH AA)CLT CLT-(DH AA)SDF},*,,,,,,,,,,,,,,,,,,,,,,,,,,,,
6,135,56,14.3,60.000000,0,0.0,56,56,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
7,136,0,14.3,7.000000,0,0.0,56,56,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
8,147,2,14.3,35.330357,28,0.0,56,56,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
9,148,1,14.3,29.517857,28,0.0,56,56,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,


In [ ]:
from datetime import date, datetime, timedelta
from collections import defaultdict
import pandas as pd


RESERVE_CODES = {"VTO", "RB", "RA", "SA", "SB", "VOR"}

def _merge_calendar_pieces(pieces, off_value=""):
    """
    Merges all text pieces for one calendar cell.

    Rules:
        - remove empty pieces
        - if there is real text and '*', drop '*'
        - if there are only '*' pieces, show one '*'
        - otherwise join real pieces with a space, not |
    """

    cleaned = [str(p).strip() for p in pieces if p and str(p).strip()]

    if not cleaned:
        return off_value

    real_pieces = [p for p in cleaned if p != "*"]

    if real_pieces:
        # Remove duplicates while preserving order
        unique_real_pieces = list(dict.fromkeys(real_pieces))
        return " ".join(unique_real_pieces)

    return "*"

def _to_date(value):
    if value is None or value == "":
        return None

    if isinstance(value, datetime):
        return value.date()

    if isinstance(value, date):
        return value

    return datetime.strptime(str(value), "%Y-%m-%d").date()


def _date_range_inclusive(start, end):
    """
    General-purpose date range.
    Allows reversed dates by correcting them.
    """

    start = _to_date(start)
    end = _to_date(end)

    if start is None or end is None:
        return []

    if end < start:
        start, end = end, start

    return [
        start + timedelta(days=i)
        for i in range((end - start).days + 1)
    ]


def _date_range_strict(start, end):
    """
    Strict date range.

    Important:
        If start > end, return [].

    This must be used for empty calendar days inside a trip.
    """

    start = _to_date(start)
    end = _to_date(end)

    if start is None or end is None:
        return []

    if start > end:
        return []

    return [
        start + timedelta(days=i)
        for i in range((end - start).days + 1)
    ]

def _format_rest(rest):
    """
    Examples:
        15.5     -> "15.5"
        "15h30"  -> "15.5"
        "10h00"  -> "10"
        "10:00"  -> "10"
    """

    if rest is None:
        return ""

    if isinstance(rest, (int, float)):
        if float(rest).is_integer():
            return str(int(rest))
        return str(round(float(rest), 1))

    text = str(rest).strip()

    if "h" in text:
        hours, minutes = text.split("h", 1)
        hours = int(hours)
        minutes = int(minutes or 0)

        value = hours + minutes / 60

        if value.is_integer():
            return str(int(value))

        return str(round(value, 1))

    if ":" in text:
        hours, minutes = text.split(":", 1)
        hours = int(hours)
        minutes = int(minutes or 0)

        value = hours + minutes / 60

        if value.is_integer():
            return str(int(value))

        return str(round(value, 1))

    return text


def _format_route_flags(route_flags):
    """
    Examples:
        None                  -> ""
        ["DH AA"]             -> "(DH AA)"
        ["DH AA", "IRO"]      -> "(DH AA,IRO)"
    """

    if not route_flags:
        return ""

    if isinstance(route_flags, str):
        flags = [route_flags]
    else:
        flags = list(route_flags)

    flags = [str(flag).strip() for flag in flags if str(flag).strip()]

    if not flags:
        return ""

    return f"({','.join(flags)})"


def _arrival_text(arrival, rest=None, close_trip=False):
    """
    Examples:
        DFW no rest      -> "DFW"
        DFW with rest    -> "[DFW15.5] "
        final SDF        -> "SDF}"
    """

    if rest is not None:
        text = f"[{arrival}{_format_rest(rest)}] "
    else:
        text = str(arrival)

    if close_trip:
        text = text.rstrip() + "}"

    return text


def _append_piece(parts_by_date, d, piece):
    if not piece:
        return

    piece = str(piece)

    # If trying to add *, only add it to a truly empty cell.
    if piece == "*":
        if not parts_by_date[d]:
            parts_by_date[d] = "*"
        return

    # If the cell currently only has *, replace it with real text.
    if parts_by_date[d] == "*":
        parts_by_date[d] = piece
        return

    if not parts_by_date[d]:
        parts_by_date[d] = piece
        return

    # Continue after rest bracket space:
    # [MDT27.8] MDT-SDF
    if parts_by_date[d].endswith(" "):
        parts_by_date[d] += piece
    else:
        parts_by_date[d] += piece

def _render_trip_by_date(assignment):
    """
    Renders one trip assignment across calendar dates.

    Example compressed same-day route:

        SDF-DFW-SDF-[ONT15.5] ONT-SDF-[DFW10]

    Rules:
        - Trip starts with {trip_id
        - Trip ends with }
        - Continuous legs do not repeat the connecting city.
        - After rest, the next leg restarts with the departure city.
        - Full empty calendar days inside a trip become *
        - Route flags go after dash and before arrival.
    """

    trip_id = assignment.get("trip_id")
    flights = assignment.get("flights") or []

    parts_by_date = defaultdict(str)

    if not flights:
        return parts_by_date

    previous_arrival = None
    previous_end_date = None
    previous_rest = None

    for index, flight in enumerate(flights):
        dep = flight.get("departure")
        arr = flight.get("arrival")

        start_date = _to_date(flight.get("start_date"))
        end_date = _to_date(flight.get("end_date"))

        if start_date is None and end_date is None:
            continue

        if start_date is None:
            start_date = end_date

        if end_date is None:
            end_date = start_date

        route_flags = _format_route_flags(flight.get("route_flags"))
        rest = flight.get("rest")

        is_first_flight = index == 0
        is_last_flight = index == len(flights) - 1

        trip_open = f"{{{trip_id}" if is_first_flight else ""

        arrival = _arrival_text(
            arr,
            rest=rest,
            close_trip=is_last_flight,
        )

        # Should we repeat the departure city?
        #
        # Do NOT repeat when:
        #   - same date
        #   - previous arrival city is same as current departure city
        #   - previous flight did NOT have rest
        #
        # DO repeat after rest:
        #   ...-SDF-[ONT15.5] ONT-SDF...
        can_compress_departure = (
            not is_first_flight
            and previous_arrival == dep
            and previous_end_date == start_date
            and previous_rest is None
            and parts_by_date[start_date]
        )

        if start_date == end_date:
            if can_compress_departure:
                # Example:
                # SDF-DFW + DFW-SDF becomes SDF-DFW-SDF
                piece = f"-{route_flags}{arrival}"
            else:
                piece = f"{trip_open}{dep}-{route_flags}{arrival}"

            _append_piece(parts_by_date, start_date, piece)

        else:
            # Flight crosses a calendar date.
            if can_compress_departure:
                departure_piece = "-"
            else:
                departure_piece = f"{trip_open}{dep}-"

            arrival_piece = f"{route_flags}{arrival}"

            _append_piece(parts_by_date, start_date, departure_piece)
            _append_piece(parts_by_date, end_date, arrival_piece)

        # Mark only FULL EMPTY calendar days between this flight and the next flight.
        # This is the only correct use of '*'.
        if index < len(flights) - 1:
            next_flight = flights[index + 1]
            next_start = _to_date(next_flight.get("start_date"))

            if next_start is not None:
                gap_start = end_date + timedelta(days=1)
                gap_end = next_start - timedelta(days=1)

                for gap_day in _date_range_strict(gap_start, gap_end):
                    _append_piece(parts_by_date, gap_day, "*")

        previous_arrival = arr
        previous_end_date = end_date
        previous_rest = rest

    return parts_by_date


def _render_code_assignment_by_date(assignment):
    """
    Handles VTO / RB / RA / SA / SB / VOR assignments.

    Example:
        {'code': 'SB', 'date': '2023-06-25'}

    Result:
        2023-06-25 -> "SB"
    """

    code = assignment.get("code")
    assignment_date = _to_date(assignment.get("date"))

    if code in RESERVE_CODES and assignment_date is not None:
        return {assignment_date: code}

    return {}


def _render_assignment_by_date(assignment):
    """
    Dispatches assignment to the correct renderer.
    """

    if assignment.get("flights"):
        return _render_trip_by_date(assignment)

    if assignment.get("code") in RESERVE_CODES:
        return _render_code_assignment_by_date(assignment)

    return {}


def master_lines_to_dataframe(
    master_lines,
    bid_start,
    bid_end,
    *,
    off_value="",
    date_col_format="%Y-%m-%d",
):
    bid_dates = _date_range_inclusive(bid_start, bid_end)
    rows = []

    for line_number, line_data in master_lines.items():
        text_by_date = defaultdict(list)

        for pp in line_data.get("PPs", line_data.get("pay_periods", [])):
            for assignment in pp.get("assignments", []):
                rendered = _render_assignment_by_date(assignment)

                for d, text in rendered.items():
                    if bid_dates[0] <= d <= bid_dates[-1]:
                        text_by_date[d].append(text)

        date_values = {}

        for d in bid_dates:
            pieces = text_by_date.get(d, [])
            date_values[d] = _merge_calendar_pieces(pieces, off_value)

        row = {
            "line_number": line_number,
            "extra_vacation_days": line_data.get("extra_vacation_days", 0),
            "training_fit_score": line_data.get("training_fit_score", 0),
            "blockiness_score": line_data.get("blockiness_score", 0),
            "tot_DO": line_data.get("tot_DO", 0),
            "company_ticket_pct": line_data.get("company_ticket_pct", 0),
        }

        for d in bid_dates:
            row[d.strftime(date_col_format)] = date_values[d]

        rows.append(row)

    df = pd.DataFrame(rows)

    if not df.empty:
        df = df.sort_values("line_number").reset_index(drop=True)

    return df

In [10]:
df = master_lines_to_dataframe(
    master_lines,
    bid_start=bid_period_info['bid_period_date_range']['start'],
    bid_end=bid_period_info['bid_period_date_range']['end'],
    off_value="",          # or "DO" if you want day-off cells marked
    date_col_format="%b %d",
)

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)  # Shows full text in cells
df

,line_number,extra_vacation_days,training_fit_score,blockiness_score,tot_DO,company_ticket_pct,DO_start_bid,DO_end_bid,May 21,May 22,May 23,May 24,May 25,May 26,May 27,May 28,May 29,May 30,May 31,Jun 01,Jun 02,Jun 03,Jun 04,Jun 05,Jun 06,Jun 07,Jun 08,Jun 09,Jun 10,Jun 11,Jun 12,Jun 13,Jun 14,Jun 15,Jun 16,Jun 17,Jun 18,Jun 19,Jun 20,Jun 21,Jun 22,Jun 23,Jun 24,Jun 25,Jun 26,Jun 27,Jun 28,Jun 29,Jun 30,Jul 01,Jul 02,Jul 03,Jul 04,Jul 05,Jul 06,Jul 07,Jul 08,Jul 09,Jul 10,Jul 11,Jul 12,Jul 13,Jul 14,Jul 15,Jul 16
0,1,2,29.3,47.482517,24,2.1,2,1,,,{7SDF-MIA-SDF},{7SDF-MIA-SDF},{14SDF-MCO-SDF},,,,,,{8SDF-MIA-SDF},{8SDF-MIA-SDF},,,,,{8SDF-MIA-SDF},{8SDF-MIA-SDF},{8SDF-MIA-SDF},{34SDF-PHL-SDF},,,,{8SDF-MIA-SDF},{8SDF-MIA-SDF},{8SDF-MIA-SDF},,,,,{22SDF-IAH-SDF},{8SDF-MIA-SDF},{8SDF-MIA-SDF},,,,,{8SDF-MIA-SDF},{8SDF-MIA-SDF},{8SDF-MIA-SDF},,,,,,,{9SDF-MIA-SDF},,,,,{9SDF-MIA-SDF},{9SDF-MIA-SDF},{9SDF-MIA-SDF},{34SDF-PHL-SDF},{46SDF-DTW-(DH DL)SDF},
1,2,2,45.3,59.150641,25,0.0,2,2,,,{4SDF-EWR-SDF},{4SDF-EWR-SDF},{4SDF-EWR-SDF},{4SDF-EWR-SDF},{31SDF-MKE-SDF},,,,,,,,,,{5SDF-EWR-SDF},{5SDF-EWR-SDF},{5SDF-EWR-SDF},,,,,{5SDF-EWR-SDF},{5SDF-EWR-SDF},{5SDF-EWR-SDF},{65SDF-EWR-SDF},,,,,,,,,,,{5SDF-EWR-SDF},{5SDF-EWR-SDF},{5SDF-EWR-SDF},{5SDF-EWR-SDF},,,,,{49SDF-JFK-SDF},{3SDF-JFK-SDF},{65SDF-EWR-SDF},,,,{6SDF-EWR-SDF},{6SDF-EWR-SDF},{6SDF-EWR-SDF},{6SDF-EWR-SDF},,
2,37,0,26.0,102.946387,28,62.5,0,1,{291SDF-,(DH DL)ATL-(DH DL)[DFW20.9],DFW-RFD-[DFW15.4],DFW-SDF-[OAK25.2],OAK-[ONT16.6],ONT-DEN-SDF},,,,,,,,,,,,,,,,{356SDF-,(DH AA)CLT-(DH AA)[DFW20.5],DFW-RFD-[DFW15.8],DFW-RFD-[DFW15.4],DFW-SDF-[OAK25.2],OAK-[ONT13.5],ONT-(DH DL)ATL-(DH DL)SDF},,,,,,,,{396SDF-(DH DL)DTW-,[LAN24.1],LAN-SDF-[LAN15.8],LAN-SDF-[LAN15.8],LAN-SDF-[LAN15.8],LAN-SDF-[LAN15.8],LAN-SDF-[EWR17.4],EWR-SDF},,,,,,,,{309SDF-(DH WN)MDW-(DH WN)[MCI20.5],MCI-SDF-[MKE15.3],MKE-SDF-[MKE15.3],MKE-SDF-[MKE15.3],MKE-SDF-[MKE15.3],MKE-SDF},
3,38,0,5.2,85.845353,31,75.0,0,14,{307SDF-,(DH WN)[ATL24.0],ATL-SDF-[CLE14.7],CLE-SDF-[CLE14.7],CLE-SDF-[CLE14.7],CLE-SDF-[CLE14.7],CLE-SDF},,,,,,,,{400SDF-(DH DL)ATL-(DH DL)[MDT27.8],*,MDT-SDF-[MDT15.2],MDT-SDF-[MDT15.2],MDT-SDF-[MDT15.2],MDT-SDF-[MDT15.9],MDT-SDF-[MDT11.6] MDT-,(DH AA)CLT-(DH AA)SDF},,,,,,,{326SDF-(DH DL)[DTW26],*,DTW-SDF-[ATL16.9],ATL-SDF-[ATL16.9],ATL-SDF-[ATL16.9],ATL-SDF-[MIA14.7],MIA-SDF},,{338SDF-(DH AA)[DFW14.8],DFW-[PHL31.6],PHL-,SDF-[ORD16.6],ORD-SDF-[MCO14.6],MCO-SDF-[CLE12.6] CLE-,(DH WN)MDW-(DH WN)SDF},,,,,,,,,,,,,,
4,105,31,63.0,71.840278,16,37.5,0,0,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,,,,{600SDF-,[HNL20.6],HNL-(DH DL)ATL-(DH DL)SDF},,,,,,{600SDF-,[HNL20.6],HNL-(DH DL)ATL-(DH DL)SDF},,,,,{601SDF-,[HNL20.6],HNL-(DH DL)ATL-(DH DL)SDF},,,,,,"{612SDF-(C,DH UPS)CGN-[FRA25.1]",FRA-,(DH EK)DXB-(DH EK)[BAH17.6] BAH-
5,106,0,9.0,87.115385,13,75.0,0,1,{395SDF-(DH DL)[ATL25.5],*,ATL-SDF-[ATL16.9],ATL-SDF-[ATL16.9],ATL-SDF-[ATL16.9],ATL-SDF-[MCO14.6],MCO-SDF-[EWR17.4],EWR-SDF},,,,,,,,,,,,,,{310SDF-,(DH AA)CLT-(DH AA)[RDU20.0],RDU-SDF-[RDU14.5],RDU-SDF-[RDU14.5],RDU-SDF-[RDU14.5],RDU-SDF-[RDU9.8] RDU-(DH AA)CLT-(DH AA)SDF},,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,
6,135,56,14.3,60.000000,0,0.0,0,1,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,
7,136,0,14.3,7.000000,0,0.0,0,1,VOR,VOR,VOR,VOR,VOR,VOR,VOR,VOR,VOR,VOR,VOR,VOR,VOR,VOR,VOR,VOR,VOR,VOR,VOR,VOR,VOR,VOR,VOR,VOR,VOR,VOR,VOR,VOR,VOR,VOR,VOR,VOR,VOR,VOR,VOR,VOR,VOR,VOR,VOR,VOR,VOR,VOR,VOR,VOR,VOR,VOR,VOR,VOR,VOR,VOR,VOR,VOR,VOR,VOR,VOR,VOR,
8,147,2,14.3,35.330357,28,0.0,2,1,,,RA,RA,RA,RA,RA,,,,RA,RA,RA,RA,,,,,,,,,,RA,RA,RA,RA,RA,,,,RA,RA,RA,RA,,,,RA,RA,RA,RA,RA,,,,,,,,,RA,RA,RA,RA,RA,
9,148,1,14.3,29.517857,28,0.0,1,8,,RA,RA,RA,RA,RA,,,,RA,RA,RA,RA,,,,RA,RA,

In [11]:
def build_line_calendar_values(line_data, bid_dates, off_value=""):
    """
    Creates the day-by-day calendar contents for one line.

    Returns:
        {
            date(2023, 6, 1): "{400SDF-ATL-[MDT27.8]",
            date(2023, 6, 2): "MDT-SDF-[MDT15.2]",
            ...
        }
    """

    RESERVE_CODES = {"VTO", "RB", "RA", "SA", "SB", "VOR"}

    def to_date(value):
        if value is None or value == "":
            return None
        if isinstance(value, datetime):
            return value.date()
        if isinstance(value, date):
            return value
        return datetime.strptime(str(value), "%Y-%m-%d").date()

    def date_range_strict(start, end):
        start = to_date(start)
        end = to_date(end)

        if start is None or end is None:
            return []

        if start > end:
            return []

        return [start + timedelta(days=i) for i in range((end - start).days + 1)]

    def format_rest(rest):
        if rest is None:
            return ""

        if isinstance(rest, (int, float)):
            return str(int(rest)) if float(rest).is_integer() else str(round(float(rest), 1))

        text = str(rest).strip()

        if "h" in text:
            hours, minutes = text.split("h", 1)
        elif ":" in text:
            hours, minutes = text.split(":", 1)
        else:
            return text

        hours = int(hours)
        minutes = int(minutes or 0)
        value = hours + minutes / 60

        return str(int(value)) if value.is_integer() else str(round(value, 1))

    def format_route_flags(route_flags):
        if not route_flags:
            return ""

        if isinstance(route_flags, str):
            flags = [route_flags]
        else:
            flags = list(route_flags)

        flags = [str(flag).strip() for flag in flags if str(flag).strip()]

        if not flags:
            return ""

        return f"({','.join(flags)})"

    def arrival_text(arrival, rest=None, close_trip=False):
        if rest is not None:
            text = f"[{arrival}{format_rest(rest)}] "
        else:
            text = str(arrival)

        if close_trip:
            text = text.rstrip() + "}"

        return text

    def append_piece(parts_by_date, d, piece):
        if not piece:
            return

        piece = str(piece)

        if piece == "*":
            if not parts_by_date[d]:
                parts_by_date[d] = "*"
            return

        if parts_by_date[d] == "*":
            parts_by_date[d] = piece
            return

        if not parts_by_date[d]:
            parts_by_date[d] = piece
            return

        if parts_by_date[d].endswith(" "):
            parts_by_date[d] += piece
        else:
            parts_by_date[d] += piece

    def render_trip(assignment):
        trip_id = assignment.get("trip_id")
        flights = assignment.get("flights") or []
        parts_by_date = defaultdict(str)

        previous_arrival = None
        previous_end_date = None
        previous_rest = None

        for index, flight in enumerate(flights):
            dep = flight.get("departure")
            arr = flight.get("arrival")

            start_date = to_date(flight.get("start_date"))
            end_date = to_date(flight.get("end_date"))

            if start_date is None and end_date is None:
                continue

            if start_date is None:
                start_date = end_date

            if end_date is None:
                end_date = start_date

            route_flags = format_route_flags(flight.get("route_flags"))
            rest = flight.get("rest")

            is_first_flight = index == 0
            is_last_flight = index == len(flights) - 1

            trip_open = f"{{{trip_id}" if is_first_flight else ""

            arrival = arrival_text(
                arr,
                rest=rest,
                close_trip=is_last_flight,
            )

            can_compress_departure = (
                not is_first_flight
                and previous_arrival == dep
                and previous_end_date == start_date
                and previous_rest is None
                and parts_by_date[start_date]
            )

            if start_date == end_date:
                if can_compress_departure:
                    piece = f"-{route_flags}{arrival}"
                else:
                    piece = f"{trip_open}{dep}-{route_flags}{arrival}"

                append_piece(parts_by_date, start_date, piece)

            else:
                if can_compress_departure:
                    departure_piece = "-"
                else:
                    departure_piece = f"{trip_open}{dep}-"

                arrival_piece = f"{route_flags}{arrival}"

                append_piece(parts_by_date, start_date, departure_piece)
                append_piece(parts_by_date, end_date, arrival_piece)

            if index < len(flights) - 1:
                next_flight = flights[index + 1]
                next_start = to_date(next_flight.get("start_date"))

                if next_start is not None:
                    gap_start = end_date + timedelta(days=1)
                    gap_end = next_start - timedelta(days=1)

                    for gap_day in date_range_strict(gap_start, gap_end):
                        append_piece(parts_by_date, gap_day, "*")

            previous_arrival = arr
            previous_end_date = end_date
            previous_rest = rest

        return parts_by_date

    def merge_pieces(pieces):
        cleaned = [str(p).strip() for p in pieces if p and str(p).strip()]

        if not cleaned:
            return off_value

        real_pieces = [p for p in cleaned if p != "*"]

        if real_pieces:
            return " ".join(dict.fromkeys(real_pieces))

        return "*"

    text_by_date = defaultdict(list)

    for pp in line_data.get("PPs", line_data.get("pay_periods", [])):
        for assignment in pp.get("assignments", []):

            if assignment.get("flights"):
                rendered = render_trip(assignment)

            elif assignment.get("code") in RESERVE_CODES:
                assignment_date = to_date(assignment.get("date"))
                rendered = {assignment_date: assignment["code"]} if assignment_date else {}

            else:
                rendered = {}

            for d, text in rendered.items():
                if bid_dates[0] <= d <= bid_dates[-1]:
                    text_by_date[d].append(text)

    return {
        d: merge_pieces(text_by_date.get(d, []))
        for d in bid_dates
    }

In [ ]:
def master_lines_to_dataframe(
    master_lines,
    bid_start,
    bid_end,
    *,
    off_value="",
    date_col_format="%Y-%m-%d",
):
    bid_dates = _date_range_inclusive(bid_start, bid_end)
    rows = []

    for line_number, line_data in master_lines.items():

        date_values = build_line_calendar_values(
            line_data,
            bid_dates,
            off_value=off_value,
        )

        row = {
            "Line Number": line_number,
            "Extra Vacation Days": line_data.get("extra_vacation_days", 0),
            "training_score": line_data.get("training_fit_score", 0),
            "Blockiness": line_data.get("blockiness_score", 0),
            "Total DO": line_data.get("tot_DO", 0),
            "% tickets paid": line_data.get("company_ticket_pct", 0),
            "Premium": line_data.get("tot_premium")
        }

        for d in bid_dates:
            row[d.strftime(date_col_format)] = date_values[d]

        rows.append(row)

    return pd.DataFrame(rows).sort_values("line_number").reset_index(drop=True)

In [13]:
df = master_lines_to_dataframe(
    master_lines,
    bid_start=bid_period_info['bid_period_date_range']['start'],
    bid_end=bid_period_info['bid_period_date_range']['end'],
    off_value="",          # or "DO" if you want day-off cells marked
    date_col_format="%b %d",
)

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)  # Shows full text in cells
df

,line_number,extra_vacation_days,training_fit_score,blockiness_score,tot_DO,company_ticket_pct,DO_start_bid,DO_end_bid,May 21,May 22,May 23,May 24,May 25,May 26,May 27,May 28,May 29,May 30,May 31,Jun 01,Jun 02,Jun 03,Jun 04,Jun 05,Jun 06,Jun 07,Jun 08,Jun 09,Jun 10,Jun 11,Jun 12,Jun 13,Jun 14,Jun 15,Jun 16,Jun 17,Jun 18,Jun 19,Jun 20,Jun 21,Jun 22,Jun 23,Jun 24,Jun 25,Jun 26,Jun 27,Jun 28,Jun 29,Jun 30,Jul 01,Jul 02,Jul 03,Jul 04,Jul 05,Jul 06,Jul 07,Jul 08,Jul 09,Jul 10,Jul 11,Jul 12,Jul 13,Jul 14,Jul 15,Jul 16
0,1,2,29.3,47.482517,24,2.1,2,1,,,{7SDF-MIA-SDF},{7SDF-MIA-SDF},{14SDF-MCO-SDF},,,,,,{8SDF-MIA-SDF},{8SDF-MIA-SDF},,,,,{8SDF-MIA-SDF},{8SDF-MIA-SDF},{8SDF-MIA-SDF},{34SDF-PHL-SDF},,,,{8SDF-MIA-SDF},{8SDF-MIA-SDF},{8SDF-MIA-SDF},,,,,{22SDF-IAH-SDF},{8SDF-MIA-SDF},{8SDF-MIA-SDF},,,,,{8SDF-MIA-SDF},{8SDF-MIA-SDF},{8SDF-MIA-SDF},,,,,,,{9SDF-MIA-SDF},,,,,{9SDF-MIA-SDF},{9SDF-MIA-SDF},{9SDF-MIA-SDF},{34SDF-PHL-SDF},{46SDF-DTW-(DH DL)SDF},
1,2,2,45.3,59.150641,25,0.0,2,2,,,{4SDF-EWR-SDF},{4SDF-EWR-SDF},{4SDF-EWR-SDF},{4SDF-EWR-SDF},{31SDF-MKE-SDF},,,,,,,,,,{5SDF-EWR-SDF},{5SDF-EWR-SDF},{5SDF-EWR-SDF},,,,,{5SDF-EWR-SDF},{5SDF-EWR-SDF},{5SDF-EWR-SDF},{65SDF-EWR-SDF},,,,,,,,,,,{5SDF-EWR-SDF},{5SDF-EWR-SDF},{5SDF-EWR-SDF},{5SDF-EWR-SDF},,,,,{49SDF-JFK-SDF},{3SDF-JFK-SDF},{65SDF-EWR-SDF},,,,{6SDF-EWR-SDF},{6SDF-EWR-SDF},{6SDF-EWR-SDF},{6SDF-EWR-SDF},,
2,37,0,26.0,102.946387,28,62.5,0,1,{291SDF-,(DH DL)ATL-(DH DL)[DFW20.9],DFW-RFD-[DFW15.4],DFW-SDF-[OAK25.2],OAK-[ONT16.6],ONT-DEN-SDF},,,,,,,,,,,,,,,,{356SDF-,(DH AA)CLT-(DH AA)[DFW20.5],DFW-RFD-[DFW15.8],DFW-RFD-[DFW15.4],DFW-SDF-[OAK25.2],OAK-[ONT13.5],ONT-(DH DL)ATL-(DH DL)SDF},,,,,,,,{396SDF-(DH DL)DTW-,[LAN24.1],LAN-SDF-[LAN15.8],LAN-SDF-[LAN15.8],LAN-SDF-[LAN15.8],LAN-SDF-[LAN15.8],LAN-SDF-[EWR17.4],EWR-SDF},,,,,,,,{309SDF-(DH WN)MDW-(DH WN)[MCI20.5],MCI-SDF-[MKE15.3],MKE-SDF-[MKE15.3],MKE-SDF-[MKE15.3],MKE-SDF-[MKE15.3],MKE-SDF},
3,38,0,5.2,85.845353,31,75.0,0,14,{307SDF-,(DH WN)[ATL24.0],ATL-SDF-[CLE14.7],CLE-SDF-[CLE14.7],CLE-SDF-[CLE14.7],CLE-SDF-[CLE14.7],CLE-SDF},,,,,,,,{400SDF-(DH DL)ATL-(DH DL)[MDT27.8],*,MDT-SDF-[MDT15.2],MDT-SDF-[MDT15.2],MDT-SDF-[MDT15.2],MDT-SDF-[MDT15.9],MDT-SDF-[MDT11.6] MDT-,(DH AA)CLT-(DH AA)SDF},,,,,,,{326SDF-(DH DL)[DTW26],*,DTW-SDF-[ATL16.9],ATL-SDF-[ATL16.9],ATL-SDF-[ATL16.9],ATL-SDF-[MIA14.7],MIA-SDF},,{338SDF-(DH AA)[DFW14.8],DFW-[PHL31.6],PHL-,SDF-[ORD16.6],ORD-SDF-[MCO14.6],MCO-SDF-[CLE12.6] CLE-,(DH WN)MDW-(DH WN)SDF},,,,,,,,,,,,,,
4,105,31,63.0,71.840278,16,37.5,0,0,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,,,,{600SDF-,[HNL20.6],HNL-(DH DL)ATL-(DH DL)SDF},,,,,,{600SDF-,[HNL20.6],HNL-(DH DL)ATL-(DH DL)SDF},,,,,{601SDF-,[HNL20.6],HNL-(DH DL)ATL-(DH DL)SDF},,,,,,"{612SDF-(C,DH UPS)CGN-[FRA25.1]",FRA-,(DH EK)DXB-(DH EK)[BAH17.6] BAH-
5,106,0,9.0,87.115385,13,75.0,0,1,{395SDF-(DH DL)[ATL25.5],*,ATL-SDF-[ATL16.9],ATL-SDF-[ATL16.9],ATL-SDF-[ATL16.9],ATL-SDF-[MCO14.6],MCO-SDF-[EWR17.4],EWR-SDF},,,,,,,,,,,,,,{310SDF-,(DH AA)CLT-(DH AA)[RDU20.0],RDU-SDF-[RDU14.5],RDU-SDF-[RDU14.5],RDU-SDF-[RDU14.5],RDU-SDF-[RDU9.8] RDU-(DH AA)CLT-(DH AA)SDF},,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,
6,135,56,14.3,60.000000,0,0.0,0,1,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,
7,136,0,14.3,7.000000,0,0.0,0,1,VOR,VOR,VOR,VOR,VOR,VOR,VOR,VOR,VOR,VOR,VOR,VOR,VOR,VOR,VOR,VOR,VOR,VOR,VOR,VOR,VOR,VOR,VOR,VOR,VOR,VOR,VOR,VOR,VOR,VOR,VOR,VOR,VOR,VOR,VOR,VOR,VOR,VOR,VOR,VOR,VOR,VOR,VOR,VOR,VOR,VOR,VOR,VOR,VOR,VOR,VOR,VOR,VOR,VOR,VOR,VOR,
8,147,2,14.3,35.330357,28,0.0,2,1,,,RA,RA,RA,RA,RA,,,,RA,RA,RA,RA,,,,,,,,,,RA,RA,RA,RA,RA,,,,RA,RA,RA,RA,,,,RA,RA,RA,RA,RA,,,,,,,,,RA,RA,RA,RA,RA,
9,148,1,14.3,29.517857,28,0.0,1,8,,RA,RA,RA,RA,RA,,,,RA,RA,RA,RA,,,,RA,RA,

In [16]:
import pandas as pd


def sort_master_lines_dataframe(
    df,
    vacation_col="extra_vacation_days",
    training_col="training_fit_score",
    blockiness_col="blockiness_score",
    days_off_col="tot_DO",
    tickets_paid_col="tickets_paid_percent",
    drop_empty_optional_cols=True,
    reset_index=True,
):
    """
    Sorts a master_lines DataFrame using dynamic priority rules.

    Baseline sort:
        1. blockiness_score high to low
        2. tot_DO high to low
        3. tickets_paid_percent high to low

    Optional priority:
        - If extra_vacation_days exists and has values > 0:
            sort by extra_vacation_days first.

        - If training_fit_score exists and has any non-zero values:
            sort by training_fit_score after vacation if vacation exists,
            otherwise before baseline.

        - If extra_vacation_days has no useful values:
            drop the column.

        - If training_fit_score is all 0:
            drop the column.
    """

    df = df.copy()

    # Required baseline columns
    required_cols = [blockiness_col, days_off_col, tickets_paid_col]
    missing_cols = [col for col in required_cols if col not in df.columns]

    if missing_cols:
        raise KeyError(f"Missing required sorting columns: {missing_cols}")

    def numeric_col(col):
        """
        Safely converts a column to numeric values for sorting.
        Missing/invalid values are treated as 0.
        """
        return pd.to_numeric(df[col], errors="coerce").fillna(0)

    # Determine if optional columns are actually useful
    vacation_active = (
        vacation_col in df.columns
        and numeric_col(vacation_col).gt(0).any()
    )

    training_active = (
        training_col in df.columns
        and numeric_col(training_col).ne(0).any()
    )

    # Drop useless optional columns
    if drop_empty_optional_cols:
        if vacation_col in df.columns and not vacation_active:
            df = df.drop(columns=[vacation_col])

        if training_col in df.columns and not training_active:
            df = df.drop(columns=[training_col])

    # Build sorting order dynamically
    sort_cols = []

    if vacation_active:
        sort_cols.append(vacation_col)

    if training_active:
        sort_cols.append(training_col)

    sort_cols.extend([
        blockiness_col,
        days_off_col,
        tickets_paid_col,
    ])

    # Create temporary numeric sort columns to avoid string sorting problems
    temp_sort_cols = []

    for col in sort_cols:
        temp_col = f"__sort_{col}"

        while temp_col in df.columns:
            temp_col += "_"

        df[temp_col] = pd.to_numeric(df[col], errors="coerce").fillna(0)
        temp_sort_cols.append(temp_col)

    df = df.sort_values(
        by=temp_sort_cols,
        ascending=[False] * len(temp_sort_cols),
        kind="mergesort",  # stable sort
    )

    df = df.drop(columns=temp_sort_cols)

    if reset_index:
        df = df.reset_index(drop=True)

    return df

In [19]:
sort_master_lines_dataframe(
    df,
    vacation_col="extra_vacation_days",
    training_col="training_fit_score",
    blockiness_col="blockiness_score",
    days_off_col="tot_DO",
    tickets_paid_col="company_ticket_pct",
    drop_empty_optional_cols=True,
    reset_index=True,
)

,line_number,extra_vacation_days,training_fit_score,blockiness_score,tot_DO,company_ticket_pct,DO_start_bid,DO_end_bid,May 21,May 22,May 23,May 24,May 25,May 26,May 27,May 28,May 29,May 30,May 31,Jun 01,Jun 02,Jun 03,Jun 04,Jun 05,Jun 06,Jun 07,Jun 08,Jun 09,Jun 10,Jun 11,Jun 12,Jun 13,Jun 14,Jun 15,Jun 16,Jun 17,Jun 18,Jun 19,Jun 20,Jun 21,Jun 22,Jun 23,Jun 24,Jun 25,Jun 26,Jun 27,Jun 28,Jun 29,Jun 30,Jul 01,Jul 02,Jul 03,Jul 04,Jul 05,Jul 06,Jul 07,Jul 08,Jul 09,Jul 10,Jul 11,Jul 12,Jul 13,Jul 14,Jul 15,Jul 16
0,135,56,14.3,60.000000,0,0.0,0,1,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,
1,105,31,63.0,71.840278,16,37.5,0,0,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,,,,{600SDF-,[HNL20.6],HNL-(DH DL)ATL-(DH DL)SDF},,,,,,{600SDF-,[HNL20.6],HNL-(DH DL)ATL-(DH DL)SDF},,,,,{601SDF-,[HNL20.6],HNL-(DH DL)ATL-(DH DL)SDF},,,,,,"{612SDF-(C,DH UPS)CGN-[FRA25.1]",FRA-,(DH EK)DXB-(DH EK)[BAH17.6] BAH-
2,172,7,14.3,29.066667,28,0.0,7,1,,,,,,,,SB,SB,SB,SB,SB,SB,SB,,,,,,,,SB,SB,SB,SB,SB,SB,SB,,,,,,,,SB,SB,SB,SB,SB,SB,SB,,,,,,,,SB,SB,SB,SB,SB,SB,SB,
3,156,7,14.3,14.533333,28,0.0,7,1,,,,,,,,SA,SA,SA,SA,SA,SA,SA,,,,,,,,SA,SA,SA,SA,SA,SA,SA,,,,,,,,SA,SA,SA,SA,SA,SA,SA,,,,,,,,SA,SA,SA,SA,SA,SA,SA,
4,2,2,45.3,59.150641,25,0.0,2,2,,,{4SDF-EWR-SDF},{4SDF-EWR-SDF},{4SDF-EWR-SDF},{4SDF-EWR-SDF},{31SDF-MKE-SDF},,,,,,,,,,{5SDF-EWR-SDF},{5SDF-EWR-SDF},{5SDF-EWR-SDF},,,,,{5SDF-EWR-SDF},{5SDF-EWR-SDF},{5SDF-EWR-SDF},{65SDF-EWR-SDF},,,,,,,,,,,{5SDF-EWR-SDF},{5SDF-EWR-SDF},{5SDF-EWR-SDF},{5SDF-EWR-SDF},,,,,{49SDF-JFK-SDF},{3SDF-JFK-SDF},{65SDF-EWR-SDF},,,,{6SDF-EWR-SDF},{6SDF-EWR-SDF},{6SDF-EWR-SDF},{6SDF-EWR-SDF},,
5,1,2,29.3,47.482517,24,2.1,2,1,,,{7SDF-MIA-SDF},{7SDF-MIA-SDF},{14SDF-MCO-SDF},,,,,,{8SDF-MIA-SDF},{8SDF-MIA-SDF},,,,,{8SDF-MIA-SDF},{8SDF-MIA-SDF},{8SDF-MIA-SDF},{34SDF-PHL-SDF},,,,{8SDF-MIA-SDF},{8SDF-MIA-SDF},{8SDF-MIA-SDF},,,,,{22SDF-IAH-SDF},{8SDF-MIA-SDF},{8SDF-MIA-SDF},,,,,{8SDF-MIA-SDF},{8SDF-MIA-SDF},{8SDF-MIA-SDF},,,,,,,{9SDF-MIA-SDF},,,,,{9SDF-MIA-SDF},{9SDF-MIA-SDF},{9SDF-MIA-SDF},{34SDF-PHL-SDF},{46SDF-DTW-(DH DL)SDF},
6,147,2,14.3,35.330357,28,0.0,2,1,,,RA,RA,RA,RA,RA,,,,RA,RA,RA,RA,,,,,,,,,,RA,RA,RA,RA,RA,,,,RA,RA,RA,RA,,,,RA,RA,RA,RA,RA,,,,,,,,,RA,RA,RA,RA,RA,
7,148,1,14.3,29.517857,28,0.0,1,8,,RA,RA,RA,RA,RA,,,,RA,RA,RA,RA,,,,RA,RA,RA,RA,RA,,,,,,,,,RA,RA,RA,RA,RA,,,,RA,RA,RA,RA,,,,RA,RA,RA,RA,RA,,,,,,,,
8,37,0,26.0,102.946387,28,62.5,0,1,{291SDF-,(DH DL)ATL-(DH DL)[DFW20.9],DFW-RFD-[DFW15.4],DFW-SDF-[OAK25.2],OAK-[ONT16.6],ONT-DEN-SDF},,,,,,,,,,,,,,,,{356SDF-,(DH AA)CLT-(DH AA)[DFW20.5],DFW-RFD-[DFW15.8],DFW-RFD-[DFW15.4],DFW-SDF-[OAK25.2],OAK-[ONT13.5],ONT-(DH DL)ATL-(DH DL)SDF},,,,,,,,{396SDF-(DH DL)DTW-,[LAN24.1],LAN-SDF-[LAN15.8],LAN-SDF-[LAN15.8],LAN-SDF-[LAN15.8],LAN-SDF-[LAN15.8],LAN-SDF-[EWR17.4],EWR-SDF},,,,,,,,{309SDF-(DH WN)MDW-(DH WN)[MCI20.5],MCI-SDF-[MKE15.3],MKE-SDF-[MKE15.3],MKE-SDF-[MKE15.3],MKE-SDF-[MKE15.3],MKE-SDF},
9,162,0,14.3,42.857143,28,0.0,0,5,RB,RB,RB,RB,,,,RB,RB,RB,RB,RB,,,,,,,,,,RB,RB,RB,RB,RB,,,,,RB,RB,RB,RB,RB,,,,,,RB,RB,RB,RB,RB,,,,RB,RB,RB,RB,,,,,
